In [ ]:
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')

# Create a folder in your Drive to hold the processed model data
save_dir = '/content/drive/MyDrive/EEG_Seizure_Project'
os.makedirs(save_dir, exist_ok=True)
print(f"Data will be saved permanently to: {save_dir}")

Connecting to Google Drive...
Mounted at /content/drive
Data will be saved permanently to: /content/drive/MyDrive/EEG_Seizure_Project


In [ ]:
import urllib.request

print("Downloading the Master Seizure Record...")
# Download the RECORDS-WITH-SEIZURES file
records_url = "https://physionet.org/files/chbmit/1.0.0/RECORDS-WITH-SEIZURES"
urllib.request.urlretrieve(records_url, "RECORDS-WITH-SEIZURES")

# Print the first few lines to prove we have it
with open("RECORDS-WITH-SEIZURES", "r") as f:
    lines = f.readlines()
    print("Files containing seizures:")
    for line in lines[:5]:
        print(f" -> {line.strip()}")

Files containing seizures:
 -> chb01/chb01_03.edf
 -> chb01/chb01_04.edf
 -> chb01/chb01_15.edf
 -> chb01/chb01_16.edf
 -> chb01/chb01_18.edf


In [ ]:
!pip install -q mne imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 34.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.


Pre-Processing and Storing Chunks of processed data on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import urllib.request
import mne
import torch
import numpy as np
from imblearn.under_sampling import RandomUnderSampler

# --- 1. SETUP ---
BASE_URL = "https://physionet.org/files/chbmit/1.0.0/"
SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'
MAX_FILES_TO_PROCESS = 150

def preprocess_signal(raw):
    raw.notch_filter(freqs=60.0, verbose=False)
    raw.filter(l_freq=1.0, h_freq=50.0, fir_design='firwin', verbose=False)
    raw.resample(sfreq=128.0, verbose=False)
    return raw

if not os.path.exists("RECORDS-WITH-SEIZURES"):
    urllib.request.urlretrieve(f"{BASE_URL}RECORDS-WITH-SEIZURES", "RECORDS-WITH-SEIZURES")

# --- 2. READ THE MASTER LIST ---
with open("RECORDS-WITH-SEIZURES", "r") as f:
    seizure_files = [line.strip() for line in f.readlines() if line.strip()]

# --- 3. THE SMART FILTER (New Text Logic) ---
pending_files = []
for file_path in seizure_files[:MAX_FILES_TO_PROCESS]:
    edf_filename = file_path.split('/')[1]
    # Only add it to the to-do list if it's NOT in Google Drive
    if not os.path.exists(f"{SAVE_DIR}/{edf_filename}_X.pt"):
        pending_files.append(file_path)

files_skipped = len(seizure_files[:MAX_FILES_TO_PROCESS]) - len(pending_files)
print(f"✅ Found {files_skipped} files already processed in Drive. Skipping them silently.")
print(f"⏳ Files left to process in this run: {len(pending_files)}\n")

# --- 4. THE LOOP ---
for count, file_path in enumerate(pending_files):
    patient_id = file_path.split('/')[0]
    edf_filename = file_path.split('/')[1]

    # Now it will print exactly how many are left in the actual remaining queue
    print(f"--- Processing {count+1}/{len(pending_files)}: {edf_filename} ---")

    try:
        summary_filename = f"{patient_id}-summary.txt"
        if not os.path.exists(summary_filename):
            urllib.request.urlretrieve(f"{BASE_URL}{patient_id}/{summary_filename}", summary_filename)

        seizure_times = []
        with open(summary_filename, "r") as sum_file:
            content = sum_file.read()
            file_block = content.split(edf_filename)[1].split('File Name')[0]
            starts = re.findall(r'Seizure(?:\s+\d+)?\s+Start Time:\s+(\d+)', file_block)
            ends = re.findall(r'Seizure(?:\s+\d+)?\s+End Time:\s+(\d+)', file_block)
            for s, e in zip(starts, ends):
                seizure_times.append((int(s), int(e)))

        print(f"  -> Found seizures at (seconds): {seizure_times}")

        print(f"  -> Downloading {edf_filename}...")
        urllib.request.urlretrieve(f"{BASE_URL}{file_path}", edf_filename)

        print("  -> Loading and filtering (this takes a moment)...")
        raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)
        raw = preprocess_signal(raw)

        data = raw.get_data()
        sfreq = 128
        window_samples = 2 * sfreq
        epochs, labels = [], []

        for start in range(0, data.shape[1] - window_samples + 1, window_samples):
            end = start + window_samples
            start_sec = start / sfreq
            end_sec = end / sfreq

            is_seizure = 0
            for sz_start, sz_end in seizure_times:
                if start_sec <= sz_end and end_sec >= sz_start:
                    is_seizure = 1
                    break

            epochs.append(data[:, start:end])
            labels.append(is_seizure)

        X, y = np.array(epochs), np.array(labels)

        print("  -> Balancing data...")
        b, c, t = X.shape
        rus = RandomUnderSampler(random_state=42)
        X_res, y_res = rus.fit_resample(X.reshape(b, c*t), y)
        X_bal = X_res.reshape(-1, c, t)

        print(f"  -> Saving {len(y_res)} balanced windows to Google Drive...")
        torch.save(torch.tensor(X_bal, dtype=torch.float32), f"{SAVE_DIR}/{edf_filename}_X.pt")
        torch.save(torch.tensor(y_res, dtype=torch.long), f"{SAVE_DIR}/{edf_filename}_y.pt")

        print(f"  -> Deleting {edf_filename} locally to clear RAM/Disk...")
        os.remove(edf_filename)
        del raw, data, X, y, X_bal, y_res
        print("  -> Done!\n")

    except Exception as e:
        print(f"  -> ERROR processing {edf_filename}: {e}")
        if os.path.exists(edf_filename):
            os.remove(edf_filename)
        print("  -> Moving to the next file...\n")

print("Pipeline complete. Check your Google Drive!")

✅ Found 92 files already processed in Drive. Skipping them silently.
⏳ Files left to process in this run: 49

--- Processing 1/49: chb07_18.edf ---
  -> Found seizures at (seconds): []
  -> Downloading chb07_18.edf...
  -> Loading and filtering (this takes a moment)...


/tmp/ipykernel_11303/2328760076.py:67: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)


  -> Balancing data...
  -> ERROR processing chb07_18.edf: The target 'y' needs to have more than 1 class. Got 1 class instead
  -> Moving to the next file...

--- Processing 2/49: chb15_54.edf ---
  -> Found seizures at (seconds): [(263, 318), (843, 1020), (1524, 1595), (2179, 2250), (3428, 3460)]
  -> Downloading chb15_54.edf...
  -> Loading and filtering (this takes a moment)...


/tmp/ipykernel_11303/2328760076.py:67: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)
/tmp/ipykernel_11303/2328760076.py:67: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4, --5
  raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)


  -> Balancing data...
  -> Saving 422 balanced windows to Google Drive...
  -> Deleting chb15_54.edf locally to clear RAM/Disk...
  -> Done!

--- Processing 3/49: chb15_62.edf ---
  -> Found seizures at (seconds): [(751, 859)]
  -> Downloading chb15_62.edf...
  -> Loading and filtering (this takes a moment)...


...

  -> Balancing data...
  -> Saving 20 balanced windows to Google Drive...
  -> Deleting chb24_15.edf locally to clear RAM/Disk...
  -> Done!

--- Processing 48/49: chb24_17.edf ---
  -> Found seizures at (seconds): [(3515, 3581)]
  -> Downloading chb24_17.edf...
  -> Loading and filtering (this takes a moment)...


/tmp/ipykernel_11303/2328760076.py:67: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)


  -> Balancing data...
  -> Saving 68 balanced windows to Google Drive...
  -> Deleting chb24_17.edf locally to clear RAM/Disk...
  -> Done!

--- Processing 49/49: chb24_21.edf ---
  -> Found seizures at (seconds): [(2804, 2872)]
  -> Downloading chb24_21.edf...
  -> Loading and filtering (this takes a moment)...


/tmp/ipykernel_11303/2328760076.py:67: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_filename, preload=True, verbose=False)


  -> Balancing data...
  -> Saving 72 balanced windows to Google Drive...
  -> Deleting chb24_21.edf locally to clear RAM/Disk...
  -> Done!

Pipeline complete. Check your Google Drive!


In [ ]:
import os
import urllib.request

SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'
BASE_URL = "https://physionet.org/files/chbmit/1.0.0/"

print("=== STARTING DATASET AUDIT ===\n")

# 1. Sanity Check: Is Google Drive actually connected?
if not os.path.exists(SAVE_DIR):
    print("🚨 ERROR: Google Drive is NOT connected, or the folder doesn't exist.")
    print("Run the 'drive.mount()' cell, and then run this script again.")
else:
    # 2. Get the Master List
    if not os.path.exists("RECORDS-WITH-SEIZURES"):
        print("Downloading RECORDS-WITH-SEIZURES...")
        urllib.request.urlretrieve(f"{BASE_URL}RECORDS-WITH-SEIZURES", "RECORDS-WITH-SEIZURES")

    with open("RECORDS-WITH-SEIZURES", "r") as f:
        # Get just the filename (e.g., 'chb01_03.edf' instead of 'chb01/chb01_03.edf')
        expected_files = [line.strip().split('/')[1] for line in f.readlines() if line.strip()]

    # 3. Check what is actually sitting in Google Drive
    found_files = []
    for file in os.listdir(SAVE_DIR):
        # We only need to check for the _X.pt files to prove it finished
        if file.endswith("_X.pt"):
            base_name = file.replace("_X.pt", "")
            found_files.append(base_name)

    # 4. Find the difference
    missing_files = [f for f in expected_files if f not in found_files]

    # 5. The Final Report
    print(f"📊 Total files required by dataset: {len(expected_files)}")
    print(f"✅ Total files safely in your Drive: {len(found_files)}")
    print(f"❌ Total files missing/pending:      {len(missing_files)}\n")

    if len(missing_files) > 0:
        print("Here are the first 10 missing files:")
        for f in missing_files[:10]: # Only print the first 10 so it doesn't flood the screen
            print(f" -> {f}")

        # This will tell you exactly where your big script will start next time
        print(f"\n💡 When you run your pipeline again, it will resume at: {missing_files[0]}")
    else:
        print("🎉 INCREDIBLE! All files are processed and sitting in your Drive!")

=== STARTING DATASET AUDIT ===

📊 Total files required by dataset: 141
✅ Total files safely in your Drive: 140
❌ Total files missing/pending:      1

Here are the first 10 missing files:
 -> chb07_18.edf

💡 When you run your pipeline again, it will resume at: chb07_18.edf


Check No. of Channels for each dataset.

In [ ]:
import os

import torch

from collections import Counter



SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'



print("🔍 Starting Channel Audit... (This will take a minute to scan all files)")



channel_counts = []

irregular_files = []



# Scan the folder

for file in os.listdir(SAVE_DIR):

    if file.endswith("_X.pt"):

        try:

            # We only need to load the X tensor to check the shape

            X_batch = torch.load(f"{SAVE_DIR}/{file}", weights_only=False) # Keep False just in case



            # The shape is (Batch, Channels, Time) -> Index 1 is Channels

            num_channels = X_batch.shape[1]

            channel_counts.append(num_channels)



            # If it's not the standard 23, save the name so we know exactly who the culprits are

            if num_channels != 23:

                irregular_files.append((file, num_channels))



        except Exception as e:

            print(f"  -> Skipping {file} due to load error.")



# Tally up the results

tally = Counter(channel_counts)



print("\n===================================")

print("📊 FINAL CHANNEL DISTRIBUTION")

print("===================================")



for channels, count in sorted(tally.items()):

    print(f" -> {count} files have {channels} channels")



if irregular_files:

    print("\n⚠️ EXACT FILES WITH IRREGULAR CHANNELS:")

    # Sort them alphabetically for easy reading

    for f, c in sorted(irregular_files):

        print(f" -> {f.replace('_X.pt', '.edf')}: {c} channels")

else:

    print("\n🎉 All files have exactly the same number of channels!"

🔍 Starting Channel Audit... (This will take a minute to scan all files)

📊 FINAL CHANNEL DISTRIBUTION
 -> 2 files have 22 channels
 -> 59 files have 23 channels
 -> 5 files have 24 channels
 -> 53 files have 28 channels
 -> 7 files have 29 channels
 -> 14 files have 38 channels

⚠️ EXACT FILES WITH IRREGULAR CHANNELS:
 -> chb04_08.edf.edf: 24 channels
 -> chb04_28.edf.edf: 24 channels
 -> chb09_06.edf.edf: 24 channels
 -> chb09_08.edf.edf: 24 channels
 -> chb09_19.edf.edf: 24 channels
 -> chb11_82.edf.edf: 28 channels
 -> chb11_92.edf.edf: 28 channels
 -> chb11_99.edf.edf: 28 channels
 -> chb12_06.edf.edf: 28 channels
 -> chb12_08.edf.edf: 28 channels
 -> chb12_09.edf.edf: 28 channels
 -> chb12_10.edf.edf: 28 channels
 -> chb12_11.edf.edf: 28 channels
 -> chb12_23.edf.edf: 28 channels
 -> chb12_27.edf.edf: 29 channels
 -> chb12_28.edf.edf: 29 channels
 -> chb12_29.edf.edf: 29 channels
 -> chb12_33.edf.edf: 29 channels
 -> chb12_36.edf.edf: 29 channels
 -> chb12_38.edf.edf: 29 channels


Top-Channel Selection


In [ ]:
import os
import torch
import numpy as np

SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'

print("🔍 Running Dynamic Channel Selection (MATLAB Translation)...")

# We will store the RMS of each channel for every file
all_session_rms = []

# Tracking counters for total transparency
total_scanned = 0
processed_count = 0
skipped_count = 0
error_count = 0

for file in os.listdir(SAVE_DIR):
    if file.endswith("_X.pt"):
        total_scanned += 1
        try:
            X_batch = torch.load(f"{SAVE_DIR}/{file}", weights_only=False)
            current_channels = X_batch.shape[1]

            # THE MASTER SLICER LOGIC
            # Only process files that have at least 23 channels to ensure consistent channel count
            if current_channels >= 20:
                # Standardize: Chop off any extra non-brain wires (24, 28, 29, 38) to 23 channels
                X = X_batch[:, :20, :]

                # Math: RMS = sqrt(mean(signal^2))
                # X shape is (Batches, Channels, Time). We calculate RMS across Time (dim=2)
                squared_signal = torch.pow(X, 2)
                mean_squared = torch.mean(squared_signal, dim=2)
                rms = torch.sqrt(mean_squared)

                # Average the RMS across all the batches in this specific file
                file_mean_rms = torch.mean(rms, dim=0)
                all_session_rms.append(file_mean_rms.numpy())

                processed_count += 1
            else:
                # Skip files with fewer than 23 channels as they cannot be standardized to 23
                skipped_count += 1

        except Exception as e:
            error_count += 1

print("\n==========================================")
print("📂 DATA SCANNING REPORT")
print("==========================================")
print(f"Total files scanned:      {total_scanned}")
print(f"✅ Successfully processed: {processed_count} files (Channels >= 23)")
print(f"⚠️ Skipped (missing data): {skipped_count} files (Channels < 23)")
print(f"❌ Load errors:            {error_count} files")

# Convert list to a numpy array: Shape will be (num_processed_files, 23_channels)
channel_rms_all = np.array(all_session_rms)

# === Your MATLAB Math exactly translated ===
eps = 1e-8
mean_rms = np.mean(channel_rms_all, axis=0)       # mean RMS per channel
std_rms  = np.std(channel_rms_all, axis=0)        # std across sessions
cv_rms   = std_rms / (mean_rms + eps)             # coefficient of variation
signal_score = mean_rms / (cv_rms + eps)          # high = strong + stable

# Sort descending to get the best ones
sorted_idx = np.argsort(signal_score)[::-1]
top5  = sorted_idx[:5]
top10 = sorted_idx[:10]

print("\n==========================================")
print("📊 FINAL CHANNEL QUALITY METRICS (0-Indexed)")
print("==========================================")
print("Calculated across the COMPLETE dataset!")
print(f"🥇 Top 5 channels:  {top5.tolist()}")
print(f"🥈 Top 10 channels: {top10.tolist()}")
print("==========================================\n")

🔍 Running Dynamic Channel Selection (MATLAB Translation)...

📂 DATA SCANNING REPORT
Total files scanned:      140
✅ Successfully processed: 140 files (Channels >= 23)
⚠️ Skipped (missing data): 0 files (Channels < 23)
❌ Load errors:            0 files

📊 FINAL CHANNEL QUALITY METRICS (0-Indexed)
Calculated across the COMPLETE dataset!
🥇 Top 5 channels:  [0, 1, 5, 2, 13]
🥈 Top 10 channels: [0, 1, 5, 2, 13, 18, 14, 8, 19, 16]



Training Model

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using Hardware: {device.type.upper()}\n")

print("1. Loading Data and Applying Dynamic Channel Selection...")
X_tensors = []
y_tensors = []
success_count, fail_count, skipped_count = 0, 0, 0

# ---> THE WINNING CHANNELS <---
# These are the Top 10 most stable channels mathematically proven across 64 sessions
BEST_CHANNELS = [0, 1, 5, 2, 13, 18, 14, 8, 19, 16]

for file in os.listdir(SAVE_DIR):
    if file.endswith("_X.pt"):
        base_name = file.replace("_X.pt", "")
        try:
            X_batch = torch.load(f"{SAVE_DIR}/{base_name}_X.pt", weights_only=False)
            y_batch = torch.load(f"{SAVE_DIR}/{base_name}_y.pt", weights_only=False)

            current_channels = X_batch.shape[1]

            if current_channels >= 20:
                # STEP 1: Standardize. If it's a 28, 29, or 38, chop off the extra non-brain wires at the end.
                X_standard = X_batch[:, :20, :]

                # STEP 2: Optimize. Now extract our mathematically proven Top 5 channels!
                X_batch = X_standard[:, BEST_CHANNELS, :]
            else:
                # We only skip the 2 files that have 22 channels (missing data)
                skipped_count += 1
                continue

            X_tensors.append(X_batch)
            y_tensors.append(y_batch)
            success_count += 1

        except Exception as e:
            fail_count += 1

print(f"\n📊 SUMMARY:")
print(f"✅ Successfully loaded & sliced to 5 channels: {success_count} files")
print(f"⚠️ Skipped irregular montages:                 {skipped_count} files")
print(f"❌ Failed to load:                             {fail_count} files\n")

print("2. Merging Data...")
X_full = torch.cat(X_tensors, dim=0)
y_full = torch.cat(y_tensors, dim=0)

print(f"-> Total Aggregated Data: X={X_full.shape}, y={y_full.shape}")
print(f"-> Total Class Breakdown: {(y_full==0).sum().item()} Normal, {(y_full==1).sum().item()} Seizure\n")

print("3. Preparing for Training...")
X_train, X_test, y_train, y_test = train_test_split(
    X_full.numpy(), y_full.numpy(), test_size=0.2, random_state=42, stratify=y_full.numpy()
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Optimized memory loaders
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("4. Building the 10-Channel EEGNet Architecture...")
class EEGNet1D(nn.Module):
    # NOTICE: num_channels is now exactly 10!
    def __init__(self, num_channels=10):
        super(EEGNet1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=num_channels, out_channels=16, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 64, 64)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

model = EEGNet1D().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("5. Training the Optimized Global Model...")
epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"-> Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

print("\n6. Final Evaluation...")
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print(f"-> Precision: {precision:.2f}")
print(f"-> Recall:    {recall:.2f}")
print(f"-> F1-Score:  {f1:.2f}")

# Save the final masterpiece
final_save_path = f"{SAVE_DIR}/GLOBAL_eeg_model_TOP10.pt"
torch.save(model.state_dict(), final_save_path)
print(f"\n🎉 OPTIMIZED Model saved to Drive: {final_save_path}")

🚀 Using Hardware: CUDA

1. Loading Data and Applying Dynamic Channel Selection...

📊 SUMMARY:
✅ Successfully loaded & sliced to 5 channels: 140 files
⚠️ Skipped irregular montages:                 0 files
❌ Failed to load:                             0 files

2. Merging Data...
-> Total Aggregated Data: X=torch.Size([12054, 10, 256]), y=torch.Size([12054])
-> Total Class Breakdown: 6027 Normal, 6027 Seizure

3. Preparing for Training...
4. Building the 10-Channel EEGNet Architecture...
5. Training the Optimized Global Model...
-> Epoch 10/50 - Loss: 0.6933
-> Epoch 20/50 - Loss: 0.6932
-> Epoch 30/50 - Loss: 0.6932
-> Epoch 40/50 - Loss: 0.6932
-> Epoch 50/50 - Loss: 0.6932

6. Final Evaluation...
-> Precision: 0.00
-> Recall:    0.00
-> F1-Score:  0.00

🎉 OPTIMIZED Model saved to Drive: /content/drive/MyDrive/EEG_Seizure_Project/GLOBAL_eeg_model_TOP10.pt


In [ ]:
# === Export trained PyTorch model to ONNX (for lightweight deployment) ===
# This creates a portable `latest.onnx` that can be served with onnxruntime (CPU).

import torch
import os

SAVE_DIR = '/content/drive/MyDrive/EEG_Seizure_Project'
onnx_drive_path = f"{SAVE_DIR}/latest.onnx"
pt_drive_path = f"{SAVE_DIR}/GLOBAL_eeg_model_TOP10.pt"

# Recreate the architecture exactly as trained
class EEGNet1D(torch.nn.Module):
    def __init__(self, num_channels=10):
        super().__init__()
        self.conv1 = torch.nn.Conv1d(in_channels=num_channels, out_channels=16, kernel_size=5, padding=2)
        self.relu1 = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool1d(kernel_size=2)

        self.conv2 = torch.nn.Conv1d(in_channels=16, out_channels=32, kernel_size=5, padding=2)
        self.relu2 = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool1d(kernel_size=2)

        self.flatten = torch.nn.Flatten()
        self.fc1 = torch.nn.Linear(32 * 64, 64)
        self.relu3 = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

export_model = EEGNet1D(num_channels=10)
export_model.load_state_dict(torch.load(pt_drive_path, map_location='cpu'))
export_model.eval()

# Model expects input shape: (batch=1, channels=10, time=256)
dummy = torch.zeros((1, 10, 256), dtype=torch.float32)

torch.onnx.export(
    export_model,
    dummy,
    onnx_drive_path,
    input_names=["eeg"],
    output_names=["logits"],
    opset_version=17,
)

print(f"✅ Exported ONNX model to: {onnx_drive_path}")
print("Next: upload latest.onnx to GCS and make it public.")


In [1]:
from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/gdrive')

# Verify the exported ONNX model exists in Google Drive
onnx_drive_path = '/content/gdrive/MyDrive/EEG_Seizure_Project/latest.onnx'

if os.path.exists(onnx_drive_path):
    print(f"✅ ONNX model found in Google Drive: {onnx_drive_path}")
else:
    print(f"❌ ONNX model NOT found at {onnx_drive_path}.")
    print("Run the training + ONNX export cells first.")


Mounting Google Drive...
Mounted at /content/gdrive
✅ Model found in Google Drive: /content/gdrive/MyDrive/EEG_Seizure_Project/GLOBAL_eeg_model_TOP10.pt


In [2]:
from google.colab import auth

print("Authenticating to Google Cloud...")
auth.authenticate_user()

Authenticating to Google Cloud...


In [1]:
# === Upload `latest.onnx` to GCS and print a public URL ===
# Make sure you already ran the auth cell: auth.authenticate_user()

project_id = 'seizuredetection420'
gcs_bucket_name = 'seizurebucket'

# Keep a stable object name so your deployed app can always fetch the newest model
gcs_object_path = 'models/latest.onnx'

onnx_drive_path = '/content/gdrive/MyDrive/EEG_Seizure_Project/latest.onnx'
gcs_uri = f"gs://{gcs_bucket_name}/{gcs_object_path}"

print(f"Setting Google Cloud project to: {project_id}")
!gcloud config set project {project_id}

print(f"Uploading ONNX model to: {gcs_uri}")
!gcloud storage cp "{onnx_drive_path}" "{gcs_uri}"

print("Making object publicly readable (if allowed by bucket policy)...")
# If this fails due to 'public access prevention' / uniform bucket-level access, you must allow public reads in the bucket settings.
!gcloud storage objects update "{gcs_uri}" --add-acl-grant=entity=AllUsers,role=READER || gsutil acl ch -u AllUsers:R "{gcs_uri}"

public_url = f"https://storage.googleapis.com/{gcs_bucket_name}/{gcs_object_path}"
print(f"\n✅ Public model URL (use this in your app as MODEL_URL):\n{public_url}")


Setting Google Cloud project to: seizuredetection420
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey

Copying model from Google Drive to GCS: gs://seizurebucket/GLOBAL_eeg_model_TOP10.pt
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
ServiceException: 401 Anonymous caller does not have storage.objects.list access to the Google Cloud Storage bucket. Permission 'storage.objects.list' denied on resource (or it may not exist).

✅ Model should now be copied to gs://seizurebucket/GLOBAL_eeg_model_TOP10.pt
